# BloodBridge AI — Demand Forecasting Model

Trains a GradientBoostingRegressor to predict blood unit demand by:
- Blood group
- Day of week
- Days since epoch (temporal trend)
- Patient transfusion frequency

Output model is deployed to SageMaker for real-time inference.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import date, timedelta
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
import joblib

plt.style.use("seaborn-v0_8-whitegrid")

df = pd.read_csv("/home/dee/Pictures/Dataset.csv")
patients = df[df["role"] == "Patient"].copy()
print(f"Patient records: {len(patients)}")

In [ ]:
# Feature engineering
records = []
epoch = date(2020, 1, 1)

for _, row in patients.iterrows():
    try:
        td = pd.to_datetime(row["expected_next_transfusion_date"]).date()
        bg = str(row.get("bridge_blood_group") or row.get("blood_group", "O Positive"))
        records.append({
            "blood_group": bg,
            "day_of_week": td.weekday(),
            "days_since_epoch": (td - epoch).days,
            "frequency_in_days": int(row.get("frequency_in_days") or 21),
            "quantity_required": int(row.get("quantity_required") or 1),
        })
    except Exception:
        pass

features = pd.DataFrame(records)
print(f"Feature rows: {len(features)}")
print(features.describe())

In [ ]:
# Train model
le = LabelEncoder()
features["bg_enc"] = le.fit_transform(features["blood_group"])

X = features[["bg_enc", "day_of_week", "days_since_epoch", "frequency_in_days"]].values
y = features["quantity_required"].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = GradientBoostingRegressor(n_estimators=100, max_depth=4, learning_rate=0.1, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
print(f"MAE: {mae:.4f}")

# Feature importance
importances = model.feature_importances_
for feat, imp in zip(["blood_group", "day_of_week", "days_since_epoch", "frequency_in_days"], importances):
    print(f"  {feat}: {imp:.3f}")

In [ ]:
# 7-day demand forecast simulation
from collections import defaultdict

today = date.today()
cutoff = today + timedelta(days=7)
demand = defaultdict(int)

for _, row in patients.iterrows():
    try:
        td = pd.to_datetime(row["expected_next_transfusion_date"]).date()
        if today <= td <= cutoff:
            bg = str(row.get("bridge_blood_group") or row.get("blood_group"))
            qty = int(row.get("quantity_required") or 1)
            demand[bg] += qty
    except Exception:
        pass

print("
7-Day Blood Demand Forecast:")
for bg, units in sorted(demand.items(), key=lambda x: -x[1]):
    print(f"  {bg}: {units} units")

plt.figure(figsize=(10, 4))
if demand:
    bgs, units = zip(*sorted(demand.items(), key=lambda x: -x[1]))
    colors = ["#ef4444" if u > 3 else "#f97316" if u > 1 else "#94a3b8" for u in units]
    plt.bar(bgs, units, color=colors)
    plt.title("7-Day Blood Demand Forecast", fontsize=13, fontweight="bold")
    plt.xticks(rotation=30)
    plt.ylabel("Units Required")
    plt.tight_layout()
    plt.show()

In [ ]:
# Save models
joblib.dump(model, "demand_model.pkl")
joblib.dump(le, "label_encoder.pkl")
print("Models saved: demand_model.pkl, label_encoder.pkl")
print("Upload to S3: aws s3 cp demand_model.pkl s3://bloodbridge-data/models/")